# Studio 1: From raw scRNA-seq counts to exploratory clusters

This teaching notebook takes one or two Cell Ranger count matrices from **data audit → QC → doublet review/removal → size-factor normalization → log transformation → PCA → neighbor graph/UMAP → Louvain, Leiden, and k-means clustering**.

> **Scientific order matters.** Scrublet uses raw counts and doublets can distort normalization, PCA, UMAP, and clustering. Therefore this notebook performs doublet scoring and the documented removal decision **before** the normalized PCA/UMAP workflow. Do not automatically discard cells merely because an algorithm labels them as doublets.

**Run cells in order.** Each code cell contains comments explaining its purpose and the result to inspect.

## Learning goals and progress checklist

By the end of this notebook, you should be able to:

- [ ] Load and combine one or two approved 10x `.h5` or `.h5ad` matrices.
- [ ] Preserve raw UMI counts and examine QC distributions before choosing filters.
- [ ] Record a dataset-specific QC and doublet decision rather than use universal cutoffs.
- [ ] Apply total-count (size-factor) normalization and `log1p` transformation.
- [ ] Identify highly variable genes, compute PCA, construct a neighbor graph, and calculate UMAP.
- [ ] Compare Louvain, Leiden, and k-means labels as **exploratory** clusterings.
- [ ] Save an `.h5ad` object with analysis parameters and provenance.

## Workflow map

```text
Raw count matrices
  → data audit and raw-count layer
  → QC metrics and documented filters
  → doublet scoring on raw counts → reviewed removal decision
  → total-count / size-factor normalization → log1p
  → highly variable genes → scaling → PCA
  → k-nearest-neighbor graph → UMAP
  → Louvain + Leiden + k-means comparison
  → saved AnnData (.h5ad) and parameter record
```

This notebook does **not** label cell types, perform batch correction, or make differential-expression claims. Those tasks require additional design decisions and evidence.

## Before you start

Create the course environment from the repository root and select **Python (scanalysis)** as the Jupyter kernel:

```bash
mamba env create -f environment.yml
conda activate scanalysis
python -m ipykernel install --user --name scanalysis --display-name "Python (scanalysis)"
jupyter lab
```

The environment includes Scanpy, Scrublet, `leidenalg`, `python-igraph`, scikit-learn, and JupyterLab. The notebook runs Louvain with Scanpy's igraph implementation. Keep protected or identifiable data out of public notebooks and AI tools.

## Step 1 — Import packages and record versions

In [ ]:
# Import data structures, analysis tools, plotting libraries, and clustering tools.
from pathlib import Path
from importlib.metadata import version

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from scipy import sparse
from sklearn.cluster import KMeans

# Make figures readable in the notebook and make stochastic steps reproducible.
sc.settings.set_figure_params(dpi=100, facecolor="white")
RANDOM_SEED = 0

# Print key versions so the analysis can be reproduced later.
for package in ["scanpy", "anndata", "numpy", "pandas", "scrublet", "igraph", "leidenalg", "scikit-learn"]:
    print(f"{package}: {version(package)}")

## Step 2 — Set input files and analysis parameters

In [ ]:
# Replace these example paths with your approved input files.
# Supported inputs: Cell Ranger filtered_feature_bc_matrix.h5 or AnnData .h5ad.
# Use one entry for a single sample or two entries for a two-sample Studio 1 exercise.
INPUT_FILES = {
    "A054": Path("data/10x_A054_cellranger/filtered_feature_bc_matrix.h5"),
    "A055": Path("data/10x_A055_cellranger/filtered_feature_bc_matrix.h5"),
}

# Set the output path now so the final object is saved in a predictable location.
OUTPUT_PATH = Path("results/studio1_preprocessed_clusters.h5ad")

# These are starter values, NOT universal rules. Inspect the plots in Step 5,
# consult the experiment design, and record why any value is changed.
MIN_GENES_PER_CELL = 200
MAX_GENES_PER_CELL = 6000
MAX_MITO_PERCENT = 20
MIN_CELLS_PER_GENE = 3

# Scrublet needs an expected doublet rate appropriate to the loading/recovery plan.
EXPECTED_DOUBLET_RATE = 0.06

# Graph/cluster settings to document and compare.
TARGET_SUM = 10_000
N_HVGS = 2_000
N_PCS = 50
N_PCS_FOR_GRAPH = 30
N_NEIGHBORS = 15
LOUVAIN_RESOLUTION = 0.8
LEIDEN_RESOLUTION = 0.8
KMEANS_N_CLUSTERS = 8

# Doublet removal is deliberately off until score/QC diagnostics are reviewed.
REMOVE_DOUBLET_CANDIDATES = False
USE_SCRUBLET_PREDICTIONS = True
CUSTOM_DOUBLET_SCORE_THRESHOLD = None  # Example: 0.25; otherwise use Scrublet's prediction.

## Step 3 — Load and combine count matrices

In [ ]:
# Define one loader so .h5 and .h5ad inputs are read consistently.
def load_count_matrix(path: Path) -> ad.AnnData:
    if not path.exists():
        raise FileNotFoundError(
            f"Input file not found: {path}. Update INPUT_FILES in Step 2 before continuing."
        )
    if path.suffix == ".h5ad":
        sample_adata = sc.read_h5ad(path)
    elif path.suffix == ".h5":
        # This expects a Cell Ranger-style 10x HDF5 count matrix.
        sample_adata = sc.read_10x_h5(path, gex_only=True)
    else:
        raise ValueError(f"Unsupported file type: {path.suffix}. Use .h5 or .h5ad input.")

    # Gene symbols can repeat in 10x data; make barcodes/gene names unambiguous.
    sample_adata.var_names_make_unique()
    return sample_adata

# Load each sample, then concatenate while keeping a clear sample label in adata.obs.
samples = [load_count_matrix(path) for path in INPUT_FILES.values()]
adata = ad.concat(
    samples,
    join="outer",
    label="sample",
    keys=list(INPUT_FILES.keys()),
    index_unique="-",
    fill_value=0,
)
adata.var_names_make_unique()

# Convert dense input to a sparse matrix when practical; count matrices are usually sparse.
if not sparse.issparse(adata.X):
    adata.X = sparse.csr_matrix(adata.X)

print(adata)
print("Cells per sample:")
print(adata.obs["sample"].value_counts().sort_index())

## Step 4 — Audit the data and preserve raw UMI counts

In [ ]:
# Inspect non-zero values to catch obvious mistakes such as negative values or prior log transforms.
nonzero_values = adata.X.data if sparse.issparse(adata.X) else np.asarray(adata.X).ravel()
if np.any(nonzero_values < 0):
    raise ValueError("Negative values found. This notebook expects non-negative raw UMI counts.")
if not np.allclose(nonzero_values, np.round(nonzero_values)):
    print("WARNING: Values are not integer-like. Confirm that the input is raw counts before Scrublet.")
else:
    print("Values are integer-like, consistent with a raw count matrix.")

# Save a raw-count layer BEFORE filtering or transformations.
# Every later transformation will leave adata.layers['counts'] unchanged for retained cells/genes.
adata.layers["counts"] = adata.X.copy()
print("Raw-count layer saved as: adata.layers['counts']")

## Step 5 — Calculate and inspect quality-control metrics

In [ ]:
# Flag mitochondrial, ribosomal, and hemoglobin genes using case-insensitive prefixes.
# Verify these rules against the species/reference used for your experiment.
gene_upper = adata.var_names.str.upper()
adata.var["mt"] = gene_upper.str.startswith("MT-")
adata.var["ribo"] = gene_upper.str.startswith(("RPS", "RPL"))
adata.var["hb"] = gene_upper.str.match(r"^HB[AB]")

# Calculate per-cell QC metrics from raw counts. This adds columns to adata.obs.
sc.pp.calculate_qc_metrics(
    adata,
    qc_vars=["mt", "ribo", "hb"],
    percent_top=None,
    log1p=False,
    inplace=True,
)

# Display QC summaries by sample before selecting filters.
qc_columns = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
print(adata.obs.groupby("sample", observed=True)[qc_columns].median())

In [ ]:
# Plot distributions. Use these diagnostics—not generic cutoffs alone—to justify QC settings.
sc.pl.violin(
    adata,
    keys=["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    groupby="sample",
    jitter=0.35,
    multi_panel=True,
)
sc.pl.scatter(adata, x="total_counts", y="pct_counts_mt", color="sample")
sc.pl.scatter(adata, x="total_counts", y="n_genes_by_counts", color="sample")

## Step 6 — Apply documented cell and gene filters

> **Decision checkpoint:** Use the Step 5 distributions, tissue expectations, library chemistry, and sample quality to justify any thresholds. Very high UMI/gene counts can indicate doublets, but they can also reflect biologically large or high-RNA cells.

In [ ]:
# Create a pass/fail record before subsetting so the rule is visible and reproducible.
adata.obs["pass_qc"] = (
    (adata.obs["n_genes_by_counts"] >= MIN_GENES_PER_CELL)
    & (adata.obs["n_genes_by_counts"] <= MAX_GENES_PER_CELL)
    & (adata.obs["pct_counts_mt"] <= MAX_MITO_PERCENT)
)
print("QC decision counts:")
print(adata.obs["pass_qc"].value_counts())

# Filter cells, then remove genes seen in too few retained cells.
adata = adata[adata.obs["pass_qc"]].copy()
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
print(f"After QC: {adata.n_obs:,} cells × {adata.n_vars:,} genes")

## Step 7 — Score doublets on raw counts

Scrublet is run **before** normalization, log transformation, PCA, UMAP, or clustering. With multiple samples, scores are calculated separately by `sample` so sample-specific loading/density is not mixed. Scrublet produces evidence; review its scores alongside QC plots and experiment-specific doublet expectations.

In [ ]:
# Select sample-aware scoring when more than one sample is present.
# The matrix in adata.X is still raw counts at this point.
scrublet_kwargs = {
    "expected_doublet_rate": EXPECTED_DOUBLET_RATE,
    "random_state": RANDOM_SEED,
}
if adata.obs["sample"].nunique() > 1:
    scrublet_kwargs["batch_key"] = "sample"

# Add 'doublet_score' and 'predicted_doublet' columns to adata.obs.
sc.pp.scrublet(adata, **scrublet_kwargs)
print(adata.obs[["doublet_score", "predicted_doublet"]].describe(include="all"))

In [ ]:
# Inspect score distributions by sample. A threshold should be defensible for this experiment.
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(
    data=adata.obs,
    x="doublet_score",
    hue="sample",
    bins=50,
    element="step",
    stat="density",
    common_norm=False,
    ax=ax,
)
ax.set_title("Scrublet doublet-score distribution")
plt.show()

# Compare score with library size; this is diagnostic evidence, not an automatic removal rule.
sc.pl.scatter(adata, x="total_counts", y="doublet_score", color="predicted_doublet")

## Step 8 — Record and, if justified, remove doublet candidates

In [ ]:
# Record the doublet definition used in this run.
# Option A: use Scrublet's predicted_doublet flag.
# Option B: supply CUSTOM_DOUBLET_SCORE_THRESHOLD after inspecting diagnostics.
if CUSTOM_DOUBLET_SCORE_THRESHOLD is None:
    if not USE_SCRUBLET_PREDICTIONS:
        raise ValueError("Choose Scrublet predictions or provide CUSTOM_DOUBLET_SCORE_THRESHOLD.")
    adata.obs["doublet_candidate"] = adata.obs["predicted_doublet"].astype(bool)
    doublet_rule = "Scrublet predicted_doublet"
else:
    adata.obs["doublet_candidate"] = adata.obs["doublet_score"] >= CUSTOM_DOUBLET_SCORE_THRESHOLD
    doublet_rule = f"doublet_score >= {CUSTOM_DOUBLET_SCORE_THRESHOLD}"

print(f"Doublet rule: {doublet_rule}")
print(adata.obs["doublet_candidate"].value_counts())

# Keep the switch False while teaching/reviewing. Set it True only after documenting the decision.
if REMOVE_DOUBLET_CANDIDATES:
    adata = adata[~adata.obs["doublet_candidate"]].copy()
    print(f"After doublet removal: {adata.n_obs:,} cells remain")
else:
    print("No cells removed yet. Set REMOVE_DOUBLET_CANDIDATES = True after your review.")

## Step 9 — Total-count (size-factor) normalization and log transformation

In [ ]:
# Calculate an explicit per-cell size factor from raw UMI totals.
# A size factor > 1 means the cell has more counts than TARGET_SUM; < 1 means fewer.
counts_per_cell = np.asarray(adata.layers["counts"].sum(axis=1)).ravel()
if np.any(counts_per_cell == 0):
    raise ValueError("Zero-count cells remain. Revisit QC filters before normalization.")

adata.obs["counts_before_normalization"] = counts_per_cell
adata.obs["size_factor"] = counts_per_cell / TARGET_SUM
print(adata.obs[["counts_before_normalization", "size_factor"]].describe())

# Normalize each cell to TARGET_SUM. Raw counts remain preserved in adata.layers['counts'].
sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
print("Median total after normalization:", np.median(np.asarray(adata.X.sum(axis=1)).ravel()))

In [ ]:
# Apply log(1 + x) to reduce the influence of a few very highly expressed genes.
sc.pp.log1p(adata)

# Store the normalized/log-transformed full gene matrix for future marker-gene visualization.
adata.raw = adata
print("adata.X is normalized + log1p; raw UMI counts remain in adata.layers['counts'].")

## Step 10 — Select highly variable genes, scale, and run PCA

In [ ]:
# Identify highly variable genes using the preserved raw-count layer.
# Seurat v3 flavor requires scikit-misc, which is included in environment.yml.
sc.pp.highly_variable_genes(
    adata,
    layer="counts",
    flavor="seurat_v3",
    n_top_genes=min(N_HVGS, adata.n_vars),
)
print(f"Highly variable genes selected: {adata.var['highly_variable'].sum():,}")
sc.pl.highly_variable_genes(adata)

In [ ]:
# Scale values for PCA while retaining unscaled normalized values in adata.raw
# and raw UMI counts in adata.layers['counts'].
sc.pp.scale(adata, max_value=10)

# Use highly variable genes for PCA but keep all genes in the AnnData object.
max_possible_pcs = min(adata.n_obs - 1, int(adata.var["highly_variable"].sum()) - 1)
n_pcs_used = min(N_PCS, max_possible_pcs)
if n_pcs_used < 2:
    raise ValueError("Too few retained cells or highly variable genes for PCA. Revisit filtering/HVG settings.")

sc.tl.pca(
    adata,
    n_comps=n_pcs_used,
    use_highly_variable=True,
    svd_solver="arpack",
    random_state=RANDOM_SEED,
)
print(f"PCA complete with {n_pcs_used} components.")

In [ ]:
# Inspect the PCA elbow plot and sample coloring before selecting PCs for the graph.
sc.pl.pca_variance_ratio(adata, n_pcs=min(n_pcs_used, 50), log=True)
sc.pl.pca(adata, color="sample")

## Step 11 — Build a neighbor graph and compute UMAP

In [ ]:
# Choose PCs based on the PCA diagnostic, not solely on this starter value.
n_pcs_for_graph_used = min(N_PCS_FOR_GRAPH, adata.obsm["X_pca"].shape[1])
sc.pp.neighbors(
    adata,
    n_neighbors=N_NEIGHBORS,
    n_pcs=n_pcs_for_graph_used,
    random_state=RANDOM_SEED,
)

# UMAP is a visualization of the neighbor graph. It does not prove discrete cell types.
sc.tl.umap(adata, random_state=RANDOM_SEED)
sc.pl.umap(adata, color="sample")

## Step 12 — Compare Louvain, Leiden, and k-means clustering

In [ ]:
# Louvain and Leiden both cluster the same neighbor graph created above.
# Louvain uses Scanpy's igraph flavor to avoid the separate legacy louvain package.
# Resolution controls granularity: larger values generally produce more clusters.
sc.tl.louvain(
    adata,
    resolution=LOUVAIN_RESOLUTION,
    random_state=RANDOM_SEED,
    flavor="igraph",
    key_added="louvain",
)
sc.tl.leiden(
    adata,
    resolution=LEIDEN_RESOLUTION,
    random_state=RANDOM_SEED,
    key_added="leiden",
)

# K-means groups cells by Euclidean distance in the selected PCA space; it does not use the graph.
if KMEANS_N_CLUSTERS > adata.n_obs:
    raise ValueError("KMEANS_N_CLUSTERS cannot exceed the number of retained cells.")
kmeans = KMeans(n_clusters=KMEANS_N_CLUSTERS, n_init=20, random_state=RANDOM_SEED)
adata.obs["kmeans"] = pd.Categorical(
    kmeans.fit_predict(adata.obsm["X_pca"][:, :n_pcs_for_graph_used]).astype(str)
)

print("Louvain clusters:", adata.obs["louvain"].nunique())
print("Leiden clusters:", adata.obs["leiden"].nunique())
print("K-means clusters:", adata.obs["kmeans"].nunique())

In [ ]:
# Visualize each clustering method on the same UMAP coordinates.
# Different methods/resolutions can disagree; investigate stability and marker evidence.
sc.pl.umap(adata, color=["louvain", "leiden", "kmeans"], wspace=0.45)

# Cross-tabulate labels to inspect agreement without claiming that one method is automatically correct.
comparison = pd.crosstab(adata.obs["leiden"], adata.obs["louvain"], normalize="index").round(2)
print("Row-normalized Leiden versus Louvain agreement:")
display(comparison)

## Step 13 — Interpretation and record of decisions

Before interpreting clusters, record the following in your notebook or lab log:

1. Input file names, reference genome, and sample/library labels.
2. QC distributions observed and the reason for every filtering threshold.
3. Expected doublet rate, the scoring/review evidence, and whether candidate doublets were removed.
4. Normalization target sum, HVG method/count, number of PCs, neighbors, and each clustering resolution.
5. Which features are stable when you change one reasonable parameter, and which conclusions should be narrowed or withheld.

> A UMAP separation or a clustering label is not a cell-type annotation or biological mechanism by itself. Use marker evidence, technical controls, and biological replication before making claims.

## Step 14 — Save the reproducible AnnData result

In [ ]:
# Store the main analysis choices inside the object before writing the file.
adata.uns["studio1_parameters"] = {
    "input_files": {sample: str(path) for sample, path in INPUT_FILES.items()},
    "qc_thresholds": {
        "min_genes_per_cell": MIN_GENES_PER_CELL,
        "max_genes_per_cell": MAX_GENES_PER_CELL,
        "max_mito_percent": MAX_MITO_PERCENT,
        "min_cells_per_gene": MIN_CELLS_PER_GENE,
    },
    "doublet_rule": doublet_rule,
    "doublet_candidates_removed": REMOVE_DOUBLET_CANDIDATES,
    "target_sum": TARGET_SUM,
    "n_hvgs_requested": N_HVGS,
    "n_pcs_used": n_pcs_used,
    "n_pcs_for_graph": n_pcs_for_graph_used,
    "n_neighbors": N_NEIGHBORS,
    "louvain_resolution": LOUVAIN_RESOLUTION,
    "leiden_resolution": LEIDEN_RESOLUTION,
    "kmeans_n_clusters": KMEANS_N_CLUSTERS,
    "random_seed": RANDOM_SEED,
}

# Create the results directory, then save the full AnnData object.
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
adata.write_h5ad(OUTPUT_PATH)
print(f"Saved: {OUTPUT_PATH.resolve()}")
print(adata)

## References

- [Scanpy preprocessing documentation](https://scanpy.readthedocs.io/en/stable/api/preprocessing.html)
- [Scanpy Louvain documentation](https://scanpy.readthedocs.io/en/stable/api/scanpy.tl.louvain.html)
- [Scanpy Leiden documentation](https://scanpy.readthedocs.io/en/stable/api/scanpy.tl.leiden.html)
- Wolock, S. L. *et al.* (2019). Scrublet: computational identification of cell doublets in single-cell transcriptomic data. *Cell Systems*.
- Traag, V. A. *et al.* (2019). From Louvain to Leiden: guaranteeing well-connected communities. *Scientific Reports*.

**Version note:** This notebook was designed for the pinned `scanalysis` environment. If packages are updated, re-run the diagnostics and record the new package versions.